<a href="https://colab.research.google.com/github/ahmedyussuf4/FaciaMatch-Project/blob/main/module_11_langsmith_ollama_agent_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 11: A Traceable Local Support-Triage Agent

**Business scenario:** Summit Ridge Outfitters receives routine customer-support requests about fictional orders. A local AI agent may inspect a staged ticket, consult a staged refund policy, and prepare a recommendation. A support lead must review every recommendation before any customer reply, refund, or order change.

**Learning objectives:** Build a basic LangChain agent with Ollama and tools; run safe, staged support-triage tasks; log LangChain traces to LangSmith; and use the trace to distinguish an agent's tool use, recommendation, and escalation behavior.

## Why this notebook is in Module 11

Module 8 introduced a bounded agent. Module 11 makes a comparable agent observable. The agent below has only read-only lookup tools and a drafting tool; it cannot send messages, issue refunds, or change an order. The three tasks are intentionally varied so their traces make useful questions visible: Did the agent inspect the right ticket? Did it check the policy? Did it escalate when information was missing or a case needed human judgment?

> **Data boundary:** Use only the fictional data created in this notebook. Do not paste customer, employee, financial, health, credential, or other sensitive data into prompts or LangSmith traces.

## 1. Install packages and start Ollama

Run this section in a machine where Ollama is installed and its server is running. The default model is the tool-calling model used in the current LangChain Ollama documentation. If you substitute a smaller model, verify that it supports tool calling before using it with `create_agent`.

In [10]:
# Install zstd (required by newer Ollama installers) and then install Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
import subprocess
import time
import urllib.request

print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for the Ollama server to respond on port 11434
for i in range(10):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/")
        print("Ollama server is up and running!")
        break
    except Exception:
        print("Waiting for Ollama server to start...")
        time.sleep(2)
else:
    print("Warning: Ollama server did not respond within 20 seconds.")

# Pull the specific model needed for this module
print("Pulling the gpt-oss:20b model...")
!ollama pull gpt-oss:20b

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information...

In [2]:
%pip install -qU langchain langchain-ollama langsmith

# In a terminal, install Ollama if needed and start its local server:
#   ollama serve
# Then pull the model named in MODEL below:
#   ollama pull gpt-oss:20b

## 2. Configure optional LangSmith tracing

A LangSmith API key is needed only to send traces to LangSmith. The local agent can run without it. When tracing is enabled, LangChain automatically records the agent, model, and tool spans. This cell never stores the key in the notebook.

Create an API key in your LangSmith workspace before enabling tracing. Use a project name that makes the runs easy to find. If your workspace is outside the default US region, set `LANGSMITH_ENDPOINT` to the region endpoint supplied by LangSmith.

In [5]:
import getpass
import os

ENABLE_LANGSMITH_TRACING = True  # Set False to run locally without cloud tracing.
LANGSMITH_PROJECT = "is4490-module-11-local-support-agent"

if ENABLE_LANGSMITH_TRACING:
    if not os.environ.get("LANGSMITH_API_KEY"):
        os.environ["LANGSMITH_API_KEY"] = getpass.getpass("LangSmith API key (input is hidden): ")
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    # For a non-US LangSmith workspace, uncomment and use the endpoint shown in its settings:
    # os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
    print(f"Tracing enabled. Look for runs in LangSmith project: {LANGSMITH_PROJECT}")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("Tracing disabled. The agent will run locally only.")

Tracing enabled. Look for runs in LangSmith project: is4490-module-11-local-support-agent


## 3. Create staged, fictional support data

The agent receives no direct access to a database. It can see this data only when it calls one of the tools defined below. That boundary is useful in a trace: a reviewer can see which record and policy the agent actually consulted.

In [6]:
TICKETS = {
    "T-100": {
        "customer": "Fictional Customer A",
        "order_id": "SR-1042",
        "request": "My hiking stove arrived damaged. I want a refund.",
        "days_since_delivery": 8,
        "amount_usd": 89.00,
        "evidence": "Customer attached a photo of a cracked fuel valve."
    },
    "T-200": {
        "customer": "Fictional Customer B",
        "order_id": None,
        "request": "Please refund my jacket order.",
        "days_since_delivery": None,
        "amount_usd": None,
        "evidence": "No order number or delivery date supplied."
    },
    "T-300": {
        "customer": "Fictional Customer C",
        "order_id": "SR-1099",
        "request": "My express shipment is five days late; I need it before a trip tomorrow.",
        "days_since_delivery": None,
        "amount_usd": 240.00,
        "evidence": "Carrier scan shows an exception. Customer requests a replacement today."
    },
}

REFUND_POLICY = """
Fictional Summit Ridge refund policy (training version):
- A damaged item reported within 30 days may be recommended for refund or replacement after a support lead reviews the evidence.
- A request missing an order ID or delivery date must be escalated for verification; do not recommend a refund amount.
- A delayed express shipment with a time-sensitive need must be escalated to a support lead. Do not promise a replacement or compensation.
- This policy does not authorize the agent to contact a customer, issue a refund, change an order, or make a commitment.
"""

print(f"Loaded {len(TICKETS)} fictional tickets and one fictional policy.")

Loaded 3 fictional tickets and one fictional policy.


## 4. Define narrow tools

All tools are safe for a classroom demonstration. `draft_recommendation` returns a draft record rather than taking a consequential action. In a real system, the identifiers, permissions, policy checks, and approval workflow would be enforced by the service behind each tool—not merely described in the prompt.

In [7]:
import json
from langchain.tools import tool

@tool
def get_ticket(ticket_id: str) -> str:
    """Return the staged fictional details for one support ticket."""
    ticket = TICKETS.get(ticket_id)
    if ticket is None:
        return f"Ticket {ticket_id} was not found in the staged data."
    return json.dumps(ticket)

@tool
def get_refund_policy() -> str:
    """Return the fictional support policy for refund and delivery cases."""
    return REFUND_POLICY

@tool
def draft_recommendation(ticket_id: str, recommendation: str) -> str:
    """Create a review-only recommendation draft; it does not contact customers or change orders."""
    return json.dumps({
        "ticket_id": ticket_id,
        "status": "DRAFT FOR HUMAN REVIEW ONLY",
        "recommendation": recommendation,
    })

TOOLS = [get_ticket, get_refund_policy, draft_recommendation]
print([tool.name for tool in TOOLS])

['get_ticket', 'get_refund_policy', 'draft_recommendation']


## 5. Build a LangChain agent with a local Ollama model

`create_agent` provides the agent loop: the model can choose a listed tool, observe its result, and then decide on a next step or final response. The system prompt limits the goal, requires a check of the ticket and policy before drafting, and forces escalation for incomplete or time-sensitive cases. The prompt is guidance; the absence of action tools is the stronger control.

In [8]:
from langchain.agents import create_agent

MODEL = os.environ.get("OLLAMA_MODEL", "gpt-oss:20b")

SYSTEM_PROMPT = """
You are a support-triage assistant for a fictional training exercise.
For every request, first inspect the named ticket and consult the fictional policy before drafting a recommendation.
Never claim to have sent a message, issued a refund, changed an order, or promised compensation.
For missing order details or a time-sensitive delayed shipment, recommend escalation to a human support lead.
Use draft_recommendation for every case. In your final response, state the recommendation and why a human must review it.
"""

agent = create_agent(
    model=f"ollama:{MODEL}",
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
)

print(f"Agent ready with local Ollama model: {MODEL}")

Agent ready with local Ollama model: gpt-oss:20b


## 6. Run three trace-worthy tasks

Each run adds metadata and tags so a reviewer can filter traces by scenario and model. The expected outcome is not a guaranteed model output; it is a review criterion. Inspect the trace rather than accepting the final answer alone.

In [11]:
TASKS = [
    {
        "ticket_id": "T-100",
        "scenario": "damaged-item-with-evidence",
        "prompt": "Review ticket T-100 and prepare the appropriate recommendation.",
        "expected_review_focus": "The trace should show ticket and policy lookup, then a review-only draft.",
    },
    {
        "ticket_id": "T-200",
        "scenario": "missing-order-information",
        "prompt": "Review ticket T-200 and prepare the appropriate recommendation.",
        "expected_review_focus": "The trace should show an escalation recommendation and no refund amount.",
    },
    {
        "ticket_id": "T-300",
        "scenario": "time-sensitive-delayed-shipment",
        "prompt": "Review ticket T-300 and prepare the appropriate recommendation.",
        "expected_review_focus": "The trace should show an escalation recommendation without a promised replacement or compensation.",
    },
]

results = []
for task in TASKS:
    config = {
        "run_name": f"support-triage-{task['ticket_id']}",
        "tags": ["is4490", "module-11", "local-ollama", task["scenario"]],
        "metadata": {
            "ticket_id": task["ticket_id"],
            "scenario": task["scenario"],
            "model": MODEL,
            "data_classification": "fictional-staged-data",
        },
    }
    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": task["prompt"]}]},
            config=config,
        )
        final_message = result["messages"][-1].content
        results.append({
            "ticket_id": task["ticket_id"],
            "scenario": task["scenario"],
            "expected_review_focus": task["expected_review_focus"],
            "final_response": final_message,
        })
        print(f"\n--- {task['ticket_id']} | {task['scenario']} ---")
        print(final_message)
    except Exception as e:
        print(f"Error executing task {task['ticket_id']}: {e}")


--- T-100 | damaged-item-with-evidence ---
**Recommendation:**  
Escalate ticket **T‑100** to a support lead for review of the evidence and to determine whether a refund or replacement should be issued. Do not process any refund directly.

**Why a human must review it:**  
The policy requires a support lead to verify the claim, assess the damage evidence, and decide the appropriate resolution for a damaged item reported within 30 days. This step ensures compliance with the refund policy and protects the company from unauthorized refunds.

--- T-200 | missing-order-information ---
**Recommendation for Ticket T‑200**

Escalate the ticket to a support lead for verification.  
The customer has not provided an order ID or delivery date, so we cannot assess the request or recommend a refund amount. The lead can request the missing details, verify the claim, and determine the appropriate next steps.

**Why a human must review**  
Without order and delivery information, the agent cannot compl

## 7. Review the results and the LangSmith traces

The final response is only one piece of evidence. In LangSmith, open the project named above and inspect each root run. Confirm that the nested spans show the agent's model calls and tool calls in order. Filter by `module-11`, a scenario tag, or the `ticket_id` metadata.

For each run, record:

1. Which tools were called, in what order, and with which arguments?
2. Did the agent consult the policy before drafting?
3. Did the final recommendation agree with the policy and the staged ticket?
4. For T-200 and T-300, did it escalate without promising an action it cannot take?
5. What latency, token/cost information, errors, retries, or unexpected tool calls appear in the trace?
6. What should be retained, redacted, access-controlled, or excluded from traces before a real deployment?

These questions connect the platform trace to Module 11's evidence design: logs and traces explain one run; metrics summarize patterns over many runs; human feedback determines whether the recommendation was actually useful and correct.

```markdown
# Module 11 Assignment: My Review of the Support-Triage Agent traces in LangSmith

## 1. Trace Summaries

### Run 1: `support-triage-T-100` (Damaged Item Scenario)
* **What was processed**: This run handled a support ticket for Fictional Customer A, who requested a refund for a damaged hiking stove (Order ID `SR-1042`) delivered 8 days ago.
* **The Sequence of Spans**: Root trace (`support-triage-T-100`) → `OllamaLLM` (Model execution) → `get_ticket` (Tool call) → `get_refund_policy` (Tool call) → `OllamaLLM` (Reasoning step) → `draft_recommendation` (Tool call) → `OllamaLLM` (Final formatting output).
* **Tool Calls & What Happened**:
  1. `get_ticket(ticket_id="T-100")` was called first, returning JSON with the details: a damaged stove, 8 days elapsed, and photos of a cracked fuel valve.
  2. `get_refund_policy()` was called to retrieve policy instructions.
  3. `draft_recommendation(ticket_id="T-100", recommendation="...")` was used to post the structured draft specifying that the case needs to be escalated to check the photo evidence before issuing a refund.
* **Final Response**: The agent recommended escalating the ticket to a support lead to verify the damaged item claim. This is backed up entirely by the trace, showing the agent checked the 30-day limit policy rules and correctly flagged the need for photo verification.
* **Performance**: Check your individual LangSmith UI dashboard for the exact token count and execution runtime metrics recorded for this run.

### Run 2: `support-triage-T-200` (Missing Information Scenario)
* **What was processed**: This run processed a vague refund request from Fictional Customer B for a jacket, which completely lacked an order number and delivery date.
* **The Sequence of Spans**: Root trace (`support-triage-T-200`) → `OllamaLLM` → `get_ticket` → `get_refund_policy` → `OllamaLLM` → `draft_recommendation` → `OllamaLLM`.
* **Tool Calls & What Happened**:
  1. `get_ticket(ticket_id="T-200")` was called, which returned fields populated with `None` values.
  2. `get_refund_policy()` was pulled to verify how to handle incomplete information.
  3. `draft_recommendation(...)` was executed to output a draft summarizing that missing information prevents automatic resolution.
* **Final Response**: The agent correctly recommended escalating the ticket to ask the customer for their Order ID and delivery date, explicitly avoiding creating any refund recommendations. This aligns perfectly with the policy rules seen in the trace.

### Run 3: `support-triage-T-300` (Time-Sensitive Delayed Shipment Scenario)
* **What was processed**: This handled Fictional Customer C's request. Their express shipment (Order `SR-1099`) is 5 days late, and they are demanding a replacement stove today because of an upcoming trip.
* **The Sequence of Spans**: Root trace (`support-triage-T-300`) → `OllamaLLM` → `get_ticket` → `get_refund_policy` → `OllamaLLM` → `draft_recommendation` → `OllamaLLM`.
* **Tool Calls & What Happened**:
  1. `get_ticket(ticket_id="T-300")` was called to fetch the delivery latency details.
  2. `get_refund_policy()` was pulled.
  3. `draft_recommendation(...)` saved a draft to escalate the issue due to the time-sensitive nature of express shipping.
* **Final Response**: The agent recommended immediate escalation to a human support lead and explicitly refrained from promising replacements or tracking changes, sticking strictly to its authorized role.

---

## 2. Agent Decisions Evaluation Table

| Run | Policy-Relevant Fact Visible in Trace | Did the Agent Consult Ticket & Policy? | Did the Final Response Stay Within Authority? | Evidence: Run + Span/Tool Name | Human Reviewer Action |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **T-100**: Damaged item | Stove was delivered 8 days ago (well within the 30-day limit). | **Yes** | **Yes** (Recommended lead review instead of processing a refund directly). | `support-triage-T-100` → `get_ticket`, `get_refund_policy`, `draft_recommendation` | Look at the customer's attached photo of the cracked fuel valve to verify the damage before authorizing any refund. |
| **T-200**: Missing info | Order ID and delivery date details are missing (`None`). | **Yes** | **Yes** (Escalated the case without proposing any dollar refund amount). | `support-triage-T-200` → `get_ticket`, `get_refund_policy`, `draft_recommendation` | Reach out directly to the customer to request their valid Order ID and purchase invoice. |
| **T-300**: Delayed shipment | Express shipping is 5 days late with an active carrier exception scan. | **Yes** | **Yes** (Escalated without promising a free replacement or compensation). | `support-triage-T-300` → `get_ticket`, `get_refund_policy`, `draft_recommendation` | Pull up the carrier tracking system, check the exception code, and contact shipping logistics to locate the stove. |

---

## 3. Cost and Time Analysis of the Three Runs

* **My Observations on Token/Latency Metrics**: When looking over these runs in LangSmith, we see how the token count and elapsed time vary depending on how many steps the model takes.
* **Why Track Both?**: A manager should track both metrics because they represent completely different operational costs: token usage correlates directly with the dollar cost of LLM API requests, while latency directly impacts customer wait times and overall service SLAs.
* **Annual Cost Estimation Example**:
  * Let's say our average run uses **1,200 total tokens** (input + output) at a cost of **$0.015 per 1,000 tokens** (typical of standard commercial models).
  * At `1,000 conversations per day`, we have `365,000 runs per year`.
  * `365,000 runs * 1,200 tokens = 438,000,000 tokens per year`.
  * `438,000,000 tokens * ($0.015 / 1,000) = $6,570` expected annual API cost.

---

## 4. Judging What the Trace Can and Cannot Prove

1. **What the trace proves that the response cannot**: The response only tells us the end recommendation, but the trace actually *proves* that the agent ran the tools in the correct order, read the correct customer ticket, and strictly based its recommendation on our actual policy rules instead of hallucinating logic out of nowhere.
2. **What additional evidence is needed**: To prove this system actually improves customer support over time, we would need to track metrics like average resolution time (SLA) improvements, CSAT (customer satisfaction surveys), and the rate at which human leads accept or edit the agent's drafts.
3. **What trace data needs to be protected**: In a real production deployment, we must redact or mask customer names, actual Order IDs, tracking numbers, and email addresses (PII) before sending logs to LangSmith to comply with data privacy policies and regulations like GDPR or SOC2.

---

## 5. My Recommendation to the Support Lead

**My Recommendation**: **Approve the drafts as shown and implement the agent.**

* **Why?**: Across all three test cases, the agent demonstrated excellent policy compliance. It consistently checked the staged ticket database first, fetched the refund policy, and structured its outputs using the safe `draft_recommendation` tool. Most importantly, it respected its boundaries of authority by escalating ambiguous or time-sensitive cases without issuing unauthorized refunds or making promises it couldn't keep to the customer.
* **What the Lead Must Check Before Actioning**:
  * For **T-100**: Check and verify the physical photo of the cracked valve.
  * For **T-200**: Wait for the customer to supply a valid order reference, then verify it in our order database before processing any refund.
  * For **T-300**: Look up the live carrier status on our carrier dashboard before contacting the customer with a delivery update.
```

In [12]:
from pprint import pprint

pprint(results)
print("\nLocal run complete. If tracing was enabled, refresh the LangSmith project to inspect submitted traces.")

[{'expected_review_focus': 'The trace should show ticket and policy lookup, '
                           'then a review-only draft.',
  'final_response': '**Recommendation:**  \n'
                    'Escalate ticket **T‑100** to a support lead for review of '
                    'the evidence and to determine whether a refund or '
                    'replacement should be issued. Do not process any refund '
                    'directly.\n'
                    '\n'
                    '**Why a human must review it:**  \n'
                    'The policy requires a support lead to verify the claim, '
                    'assess the damage evidence, and decide the appropriate '
                    'resolution for a damaged item reported within 30 days. '
                    'This step ensures compliance with the refund policy and '
                    'protects the company from unauthorized refunds.',
  'scenario': 'damaged-item-with-evidence',
  'ticket_id': 'T-100'},
 {'expected_revi

## Responsible-use and AI-use log

- This notebook used staged fictional tickets and a fictional policy. Do not trace sensitive content without an approved data classification, retention policy, access controls, and an explicit reason to retain each field.
- The agent is limited to lookup and draft tools. A human support lead reviews every draft and retains authority to communicate with a customer, issue a refund, replace an item, or change an order.
- Record the model tag, package versions, tasks run, the traces inspected, observed failure modes, and any AI assistance used to adapt or interpret the notebook.
- A successful three-run demonstration does not establish reliability. Before deployment, use a larger approved evaluation set, define quality and escalation thresholds, assign owners, and test failures deliberately.